In [1]:
# 导入必要的库
import pandas as pd
import numpy as np
import re
import os
import jieba  # 用于中文分词

In [2]:
# 初始化jieba分词器
jieba.initialize()

Building prefix dict from the default dictionary ...
Dumping model to file cache C:\Users\tzcha\AppData\Local\Temp\jieba.cache
Loading model cost 0.868 seconds.
Prefix dict has been built successfully.


In [3]:
# 定义非普通用户关键词（可根据实际情况扩展）
NON_USER_KEYWORDS = [
    '公司', '集团', '企业', '有限公司', '股份', '官方', '政府', '政务',
    '机构', '组织', '委员会', '协会', '商会', '中心', '平台', '网站',
    '网店', '商城', '旗舰店', '专营店', '直营店', '代理', '工作室', '传媒',
    '广告', '推广', '营销', '运营', '服务', '客服', '热线', '公众号',
    '微博', '平台', '博主', '大V', '网红', '媒体', '记者', '新闻',
    '车管', '医院', '券', '优惠券', '折扣券', '车管所', '交警', '医生',
    '护士', '医师', '医疗', '保健', '健康', '体检', '门诊', '药店',
    '药房', '优惠', '促销', '活动', '福利', '礼品', '赠品', '会员',
    'VIP', '专属', '特权', '专享', '限时', '限量', '抢购', '秒杀',
    '白菜', '白菜价', '白菜券', '达人', '分享达人', '种草', '买买', '团购',
    '环保', '除甲醛', '空气净化', '环境监测', '空气质量','凤凰网','领导',
    '石家庄', '唐山', '秦皇岛', '邯郸', '邢台', '保定', '张家口', '承德', '沧州', '廊坊', '衡水', '太原', '大同', '阳泉', '长治', '晋城', '朔州', '晋中', '运城', '忻州', '临汾', '吕梁', '呼和浩特', '包头', '乌海', '赤峰', '通辽', '鄂尔多斯', '呼伦贝尔', '巴彦淖尔', '乌兰察布', '沈阳', '大连', '鞍山', '抚顺', '本溪', '丹东', '锦州', '营口', '阜新', '辽阳', '盘锦', '铁岭', '朝阳', '葫芦岛', '长春', '吉林', '四平', '辽源', '通化', '白山', '松原', '白城', '哈尔滨', '齐齐哈尔', '鸡西', '鹤岗', '双鸭山', '大庆', '伊春', '佳木斯', '七台河', '牡丹江', '黑河', '绥', '南京', '无锡', '徐州', '常州', '苏州', '南通', '连云港', '淮安', '盐城', '扬州', '镇江', '泰州', '宿迁', '杭州', '宁波', '温州', '嘉兴', '湖州', '绍兴', '金华', '衢州', '舟山', '台州', '丽水', '合肥', '芜湖', '蚌埠', '淮南', '马鞍山', '淮北', '铜陵', '安庆', '黄山', '阜阳', '宿州', '滁州', '六安', '宣城', '池州', '亳州', '福州', '厦门', '莆田', '三明', '泉州', '漳州', '南平', '龙岩', '宁德', '南昌', '景德镇', '萍乡', '九江', '抚州', '鹰潭', '赣州', '吉安', '宜春', '新余', '上饶', '济南', '青岛', '淄博', '枣庄', '东营', '烟台', '潍坊', '济宁', '泰安', '威海', '日照', '临沂', '德州', '聊城', '滨州', '菏泽', '郑州', '开封', '洛阳', '平顶山', '安阳', '鹤壁', '新乡', '焦作', '濮阳', '许昌', '漯河', '三门峡', '南阳', '商丘', '信阳', '周口', '驻马店', '武汉', '黄石', '十堰', '宜昌', '襄阳', '鄂州', '荆门', '孝感', '荆州', '黄冈', '咸宁', '随州', '长沙', '株洲', '湘潭', '衡阳', '邵阳', '岳阳', '常德', '张家界', '益阳', '郴州', '永州', '怀化', '娄底', '广州', '韶关', '深圳', '珠海', '汕头', '佛山', '江门', '湛江', '茂名', '肇庆', '惠州', '梅州', '汕尾', '河源', '阳江', '清远', '东莞', '中山', '潮州', '揭阳', '云浮', '南宁', '柳州', '桂林', '梧州', '北海', '防城港', '钦州', '贵港', '玉林', '百色', '贺州', '河池', '来宾', '崇左', '海口', '三亚', '三沙', '儋州', '成都', '自贡', '攀枝花', '泸州', '德阳', '绵阳', '广元', '遂宁', '内江', '乐山', '南充', '眉山', '宜宾', '广安', '达州', '雅安', '巴中', '资阳', '贵阳', '六盘水', '遵义', '安顺', '毕节', '铜仁', '昆明', '曲靖', '玉溪', '保山', '昭通', '丽江', '普洱', '临沧', '拉萨', '日喀则', '昌都', '林芝', '山南', '那曲', '西安', '铜川', '宝鸡', '咸阳', '渭南', '延安', '汉中', '榆林', '安康', '商洛', '兰州', '嘉峪关', '金昌', '白银', '天水', '武威', '张掖', '平凉', '酒泉', '庆阳', '定西', '陇南', '西宁', '海东', '银川', '石嘴山', '吴忠', '固原', '中卫', '乌鲁木齐', '克拉玛依', '吐鲁番', '哈密'# 添加与空气相关的关键词
]


In [4]:
# 添加自定义词典
for word in NON_USER_KEYWORDS:
    jieba.add_word(word, freq=10000)  # 添加 freq 参数

In [5]:
def is_ordinary_user(name):
    """
    判断是否为普通用户（非公司、政府等账户）
    现在基于Name列进行判断
    
    参数:
    name: 用户名文本
    
    返回:
    True: 普通用户
    False: 非普通用户（公司、政府等）
    """
    if not isinstance(name, str) or name.strip() == "":
        return True
    
    # 使用jieba进行分词 - 使用cut方法并转换为列表
    words = list(jieba.cut(name))
    
    # 检查是否包含非普通用户关键词
    for word in words:
        if word in NON_USER_KEYWORDS:
            return False
    
    # 检查关键词组合（如"环保"+"除甲醛"）
    for i in range(len(words) - 1):
        bigram = words[i] + words[i+1]
        if bigram in NON_USER_KEYWORDS:
            return False
    
    return True

In [6]:
def clean_office_data(df, max_content_length=500):
    """
    清洗办公室空气数据：
    1. 删除重复内容
    2. 剔除非普通用户（基于Name列）
    3. 删除文字过长的内容
    4. 删除发表超过2个相关博文的用户
    5. 删除Content列中带"【"和"】"的整行内容
    
    
    参数:
    input_file: 输入文件路径
    output_file: 输出文件路径
    max_content_length: 内容最大长度阈值（默认500字符）
    """

    
    # 1. 删除完全重复的行
    initial_count = len(df)
    df = df.drop_duplicates()
    print(f"删除完全重复行后: {len(df)} (删除了 {initial_count - len(df)} 行)")
    
    # 2. 剔除非普通用户（基于Name列）
    if 'Name' in df.columns:
        initial_count = len(df)
        
        # 添加调试信息
        print("\n开始基于Name列识别非普通用户...")
        
        # 过滤非普通用户
        df['IsOrdinary'] = df['Name'].apply(is_ordinary_user)
        
        # 收集被删除的非普通用户信息
        non_ordinary_df = df[~df['IsOrdinary']]
        print(f"发现 {len(non_ordinary_df)} 个非普通用户")
        
        # 输出一些被删除的用户名样本
        if len(non_ordinary_df) > 0:
            print("\n被删除的非普通用户名样本:")
            for i, row in non_ordinary_df.head(10).iterrows():
                print(f"- {row['Name']}")
        
        # 过滤掉非普通用户
        df = df[df['IsOrdinary']]
        print(f"剔除非普通用户后: {len(df)} (删除了 {initial_count - len(df)} 行)")
        # 删除临时列
        df = df.drop(columns=['IsOrdinary'])
    
    # 3. 删除Content列中带"【"和"】"的整行内容
    if 'Content' in df.columns:
        initial_count = len(df)
        
        # 检查Content列是否包含"【"或"】"
        pattern = r'【.*】|【|】'  # 匹配【】及其内容，或单独的【或】
        mask = df['Content'].astype(str).str.contains(pattern, na=False)
        
        # 统计要删除的行数
        to_delete_count = mask.sum()
        print(f"发现 {to_delete_count} 行内容包含【或】")
        
        # 输出一些将被删除的内容样本
        if to_delete_count > 0:
            print("\n将被删除的内容样本:")
            for i, row in df[mask].head(5).iterrows():
                print(f"- {row['Content'][:100]}...")  # 只显示前100个字符
        
        # 删除包含【或】的行
        df = df[~mask]
        print(f"删除包含【或】的内容后: {len(df)} (删除了 {initial_count - len(df)} 行)")
    
    
    if 'Content' in df.columns:
        initial_count = len(df)
        
        pattern = r'《.*》'  # 匹配《》及其内容
        mask = df['Content'].astype(str).str.contains(pattern, na=False)
        
        # 统计要删除的行数
        to_delete_count = mask.sum()
        print(f"\n\n发现 {to_delete_count} 行内容包含《或》")
        
        # 输出一些将被删除的内容样本
        if to_delete_count > 0:
            print("\n将被删除的内容样本:")
            for i, row in df[mask].head(5).iterrows():
                print(f"- {row['Content'][:100]}...")  # 只显示前100个字符
        
        # 删除包含【或】的行
        df = df[~mask]
        print(f"删除包含《》的内容后: {len(df)} (删除了 {initial_count - len(df)} 行)")

    if 'Content' in df.columns:
        initial_count = len(df)
        
        pattern = r'小说|主角|章节|全集|完整版|全文|大结局|完结|节选'
        mask = df['Content'].astype(str).str.contains(pattern, na=False)
        
        # 统计要删除的行数
        to_delete_count = mask.sum()
        print(f"\n\n发现 {to_delete_count} 行内容包含小说内容")
        
        # 输出一些将被删除的内容样本
        if to_delete_count > 0:
            print("\n将被删除的内容样本:")
            for i, row in df[mask].head(5).iterrows():
                print(f"- {row['Content'][:100]}...")  # 只显示前100个字符
        
        df = df[~mask]
        print(f"删除包含“小说”的内容后: {len(df)} (删除了 {initial_count - len(df)} 行)")
        

    if 'Content' in df.columns:
        # 计算每条内容的长度
        content_lengths = df['Content'].astype(str).apply(len)
        
        # 统计过长内容的数量
        too_long_count = sum(content_lengths > max_content_length)
        print(f"内容超过 {max_content_length} 字符的行数: {too_long_count}")
        
        
        df = df[content_lengths <= max_content_length]
        # 过滤掉过短的内容
        df = df[content_lengths >= 10]
        print(f"删除过长内容后: {len(df)} 行")
    

    if 'Content' in df.columns:
        # 基于内容和名称删除重复（保留第一条）
        initial_count = len(df)
        df = df.drop_duplicates(subset=['Content'], keep='first')
        print(f"基于内容和名称删除重复后: {len(df)} (删除了 {initial_count - len(df)} 行)")
    

    if 'Name' in df.columns:
        # 统计每个用户的博文数量
        user_post_counts = df['Name'].value_counts().reset_index()
        user_post_counts.columns = ['Name', 'PostCount']
        
        # 找出发表超过2篇博文的用户
        frequent_posters = user_post_counts[user_post_counts['PostCount'] >= 2]['Name'].tolist()
        print(f"发现 {len(frequent_posters)} 个用户发表超过2篇博文")
        
        # 输出一些频繁发帖用户样本
        if len(frequent_posters) > 0:
            print("\n频繁发帖用户样本:")
            for name in frequent_posters[:10]:
                print(f"- {name}")
        
        # 删除这些用户的所有博文
        initial_count = len(df)
        df = df[~df['Name'].isin(frequent_posters)]
        print(f"删除频繁发帖用户后: {len(df)} (删除了 {initial_count - len(df)} 行)")
    
    # 7. 处理缺失值（可选）
    # 删除所有列都为NaN的行
    initial_count = len(df)
    df = df.dropna(how='all')
    print(f"删除全空行后: {len(df)} (删除了 {initial_count - len(df)} 行)")
    
    return df

In [7]:
def clean_text(text: str, 
                   remove_url: bool = True,
                   remove_mentions: bool = True,
                   remove_topics: bool = True,  # 话题通常包含重要信息
                   keep_emoji: bool = True,
                   remove_numbers: bool = True,
                   remove_repeat_chars: bool = True) -> str:
    """
    基础文本清洗
    
    Args:
        text: 原始文本
        remove_url: 是否移除URL
        remove_mentions: 是否移除@提及
        remove_topics: 是否移除话题标签
        keep_emoji: 是否保留表情符号
        remove_numbers: 是否移除数字
        remove_repeat_chars: 是否移除重复字符
    """
    patterns = {'url': re.compile(r'https?://\S+'),
            'user_mention': re.compile(r'@[\w\u4e00-\u9fff_-]+'),
            'topic': re.compile(r'#([^#]+?)#'),
            'emoji': re.compile(r'\[[\w\u4e00-\u9fff]+\]'), 
            'numbers': re.compile(r'\d+'),
            'repeat_chars': re.compile(r'(.)\1{2,}')}
    if not isinstance(text, str):
        return ""
    text = text.strip()
    
    # 处理URL
    if remove_url:
        text = patterns['url'].sub(' ', text)
    
    # 处理@提及
    if remove_mentions:
        text = patterns['user_mention'].sub(' ', text)
    
    # 处理话题标签（可以选择移除#符号但保留内容）
    if remove_topics:
        text = patterns['topic'].sub(' ', text)
    else:
        # 只移除#号，保留话题内容
        text = re.sub(r'#', '', text)
    
    # 处理表情符号
    if not keep_emoji:
        text = patterns['emoji'].sub(' ', text)
        # 移除unicode表情
        text = emoji.replace_emoji(text, replace=' ')
    
    # 处理数字
    if remove_numbers:
        text = patterns['numbers'].sub(' ', text)
    
    # 处理重复字符（如"哈哈哈"->"哈"）
    if remove_repeat_chars:
        text = patterns['repeat_chars'].sub(r'\1', text)
    
    # 移除多余空白字符
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

In [8]:
stopwords= pd.read_csv(".\\stopwords.txt",encoding= 'utf8',index_col=False)
stopwords=stopwords['stopword'].values.tolist()

In [9]:
def remove_stopwords(text,stopwords):
    for word in stopwords:
        #print('正在处理',word)
        pattern=re.compile(word)
        text = pattern.sub(' ', text)
    return text.strip()

In [11]:
def filter_float_lists(data):
    """
    判断列表中的元素是否为float，并删除长度较短的内容
    data: 包含多个列表的列表
    返回：过滤后的结果
    """
    # 第一步：筛选出所有元素都是float的列表

    return str(data)

In [12]:
#path='.\\air_quality\\办公室空气_全部年份合并clean.csv'
#path='.\\vent\\办公室通风16-25.csv'
#path='.\\other\\办公室吵202425.csv'
#path='zaoyin\\办公室噪声201625.csv'
#'.\\Temp\\办公室温度2122232425_clean.csv'
#'.\\Window\\办公室窗1819202122232425_all_clean.csv'

In [13]:
path='.\\Window\\'
name='EnvPrbAll.csv'

In [14]:
df=pd.read_csv(path+name)

In [15]:
#df['Content']=df['Content'].apply(filter_float_lists)
df['Content']=df['Content'].astype(str)

In [16]:
df['Content']=df['Content'].apply(clean_text)

In [17]:
df['Content']=df['Content'].apply(lambda text:remove_stopwords(text,stopwords))

In [18]:
# 执行数据清洗
cleaned_data = clean_office_data(df, max_content_length=1500)

删除完全重复行后: 228809 (删除了 0 行)

开始基于Name列识别非普通用户...
发现 12411 个非普通用户

被删除的非普通用户名样本:
- 大曲微博
- 吐槽老王和老王家傻儿子专用微博
- 小虎的微博呀
- 兰州小记小小苏
- 杭州微博城事
- 生态美家除甲醛机构
- 林间朝阳1
- 千里眼西安
- 西安新鲜事
- 广州维秘医疗美容
剔除非普通用户后: 216398 (删除了 12411 行)
发现 17987 行内容包含【或】

将被删除的内容样本:
- 【高三  子总 充满希望】去 夏天，清晨早自习，同学们坐 教室 手捧 书，大声朗读  某一刻，L同学 知道什么原因趴 桌子上睡 起来，或许读累 ？一直  状态，点名提醒抽背她早自习 早读任务，没有背会...
- 【春秀打官司】（ 集电视连续剧）编剧 胡定凯 导演 调皮儿伯格 老谋子 摄影 老谋子 主演 马若英（饰龚凰律师）甄梅婷（饰春秀）贾孙俪（饰女儿）大傻（饰厂长）贾志伟（饰东方集团董事长）马德华（饰国资委...
- 【警官 】老江组织上从 龄考虑，对今  岁 老江 工作进行 调整，让他离开  们 科室，到另外  科室工作去  长期从事培训学员 常管理工作 江老师尽管有太多  舍， 只能服从组织 安排，没有过多 告...
- 顾临钧沐箐箐顾临钧沐箐箐（顾临钧沐箐箐 ）（已    ）   阁❗  《顾临钧沐箐箐》❗   顾临钧，沐箐箐 请到公『众』号【小雨 】回书号  ， 行 “箐箐， 很想   ”顾临钧低声喃语， 过声音却...
- 【内天井加梯 居委会专户监管  上海这项民心工程 断解难题 破瓶颈】既有多层住宅加装电梯 重  民心工程，解决悬空老人上下楼 难题 近 来，本市加梯工作持续按下加速键， 断攻克技术 金融方面 难题，今...
删除包含【或】的内容后: 198411 (删除了 17987 行)


发现 5222 行内容包含《或》

将被删除的内容样本:
- 《泡泡队变动物 无意义童话故事》米葵和韩帅开 家叫愿来   小店，主打咨询服务和愿望实现业务，一 大概接客十次，勉强糊口 至于这怎么能做到糊口 ，韩帅拉开小店后门，看， 们有菜地 韩帅眨巴 大眼睛介绍...
- 宋梓柠司珩宋梓柠司珩(抖音热推  )宋梓柠司珩    阁❗  《宋梓柠司珩》❗  宋梓柠，司珩 请到公『众』号〖小

In [19]:
import time
cleaned_data.to_csv(path+str(time.time())[-6:]+'Cleaned'+name)

In [49]:
import jieba.analyse
def count_phrase_occurrences(text, phrase):  
    """  
    统计词组在字符串中出现的次数  
  
    :param text: 待搜索的字符串  
    :param phrase: 要统计的词组  
    :return: 词组在字符串中出现的次数  
    """  
    return text.count(phrase)

In [50]:
content = " ".join(df["Content"].dropna())
tags = jieba.analyse.extract_tags(content, topK=200, withWeight=False)
stopwords = pd.read_csv("stopwords.txt",encoding= 'utf8',index_col=False)
stopwords_list=stopwords['stopword'].values
new_tags=[]
for tag in tags:
    if tag not in stopwords_list:
        new_tags.append(tag)
text =" ".join(new_tags)
print(text)

通风 空调 开窗 流感 健康 冬季 噪声 同事 室内 天气 污染 装修 感冒 工作 环境 甲醛 窗户 每天 感觉 安全 冬天 自己 取暖器 夏季 口罩 二手烟 养生 视频 上班 小时 房间 病毒 可以 预防 空间 湿气 知道 这个 空气 发布 高温 可能 没有 最近 早上 应急 味道 因为 打开 影响 长期 生活 大家 设计 发现 预警 什么 能防 身体 直接 时候 工位 公司 宜光 宜多食 密闭 有益健康 中招 长时间 宜多 时间 抽烟 觉得 中毒 除甲醛 液化气 员工 出租 防护 一氧化碳 睡觉 宜喝 温暖 夏天 提醒 洗澡 耳机 使用 气温 避免 多喝水 咳嗽 容易 围巾 导致 开始 实验室 肾结石 常识 凉水 已经 噪音 出现 座椅 呼吸道 突然 洗脚 苦味 噪声污染 风扇 选择 厂房 有损 写字楼 怎么 问题 温度 雾气 网友 空气质量 提示 烟味 下午 症状 老板 吸烟 朋友 金桥 以下 上身 注意 豆浆 一下 山东 中午 喜欢 下班 领导 关于 流感病毒 地铁口 橙色 午休 立冬 特别 风险 采光 走廊 宿舍 很多 停留 然后 如何 最全 生理 单位 缺氧 保持 企业 次数 做好 一直 传播 发烧 一天 这些 精装 学校 地瓜 部门 舒适 结果 位置 委员会 起来 发生 建议 风水 为什么 发个 阳光 有人 徐先生 男子 适合 通勤 持续 生产 声音 其实 地方 一些 太冷 地铁 通知 空气净化 绿植 原因 市民 介绍


In [47]:
i=1
word_dict={}
for tag in new_tags:
    if tag not in stopwords and i<=100:
        num_count=count_phrase_occurrences(content,tag)
        word_dict[tag]=num_count
    i+=1
print(word_dict)

{'通风': 3101, '空调': 1304, '开窗': 842, '流感': 1065, '健康': 1242, '冬季': 814, '噪声': 775, '同事': 646, '室内': 789, '天气': 796, '污染': 982, '装修': 654, '感冒': 586, '工作': 1263, '环境': 1274, '甲醛': 578, '窗户': 517, '每天': 659, '感觉': 595, '安全': 818, '冬天': 463, '自己': 952, '取暖器': 263, '夏季': 437, '口罩': 333, '二手烟': 240, '养生': 323, '视频': 351, '上班': 441, '小时': 530, '房间': 381, '病毒': 593, '可以': 715, '预防': 388, '空间': 518, '湿气': 238, '知道': 595, '这个': 655, '空气': 870, '发布': 482, '高温': 330, '可能': 624, '没有': 780, '最近': 379, '早上': 312, '应急': 266, '味道': 289, '因为': 519, '打开': 368, '影响': 532, '长期': 414, '生活': 497, '大家': 426, '设计': 425, '发现': 489, '预警': 251, '什么': 812, '能防': 163, '身体': 399, '直接': 390, '时候': 531, '工位': 161, '公司': 695, '宜光': 155, '宜多食': 154, '密闭': 195, '有益健康': 167, '中招': 186, '长时间': 265, '宜多': 307, '时间': 838, '抽烟': 227, '觉得': 369, '中毒': 273, '除甲醛': 133, '液化气': 175, '员工': 249, '出租': 233, '防护': 224, '一氧化碳': 187, '睡觉': 256, '宜喝': 140, '温暖': 243, '夏天': 243, '提醒': 274, '洗澡': 201, '耳机': 186, '使用': 396, '气温': 249, '避免'

In [14]:
content_array=df['Content'].values

In [15]:
#keywords=['香烟','吸烟','烟味','二手烟','抽烟','烟雾']
#keywords=['发烧','感冒','难受','头晕','头疼','安眠药']
keywords=['冷死','不冷','太冷','冷风']
#keywords=['不透气','甲醛','味道','污染']
i=0
for content in content_array:
    for keyword in keywords:
        if keyword in str(content):
            i+=1
            break
print(f'办公室噪声数据集{keywords},{i/len(content_array)}') 

办公室噪声数据集['冷死', '不冷', '太冷', '冷风'],0.002167238576846668
